# security-universe — resolving every security type

`security-universe`'s resolvers answer one question: **what instrument is this?**
Given a vendor/native symbol they return an enriched `Security` with a stable,
vendor-neutral `security_id` (and richer fields: expiry, strike, AM/PM session,
settlement, contract multiplier).

They are **pure, offline, deterministic** — no network, no market data. This
notebook resolves one symbol of every supported family through a single
`ResolverChain`.

| Family | `security_id` shape | Self-detects? |
|---|---|---|
| Stock / ETF | `equity:AAPL` | no (needs `security_type`) |
| Listed option (OCC) | `option:SPXW:2026-06-19:put:6100` | yes |
| Future | `future:ES:M6` | no |
| **Option on future** | `future_option:ES:U6:2026-07-01:call:6325` | yes |
| Crypto | `crypto:BTC-USD` | no |


In [1]:
from security_universe import Security, SecurityType
from security_universe.resolvers import ResolverChain

chain = ResolverChain.default()
print("members:", [type(r).__name__ for r in chain.resolvers])

members: ['FutureOptionSecurityIdResolver', 'OCCSecurityIdResolver', 'EquitySecurityIdResolver', 'FutureSecurityIdResolver', 'CryptoSecurityIdResolver']


In [2]:
FIELDS = [
    "security_type", "security_id", "short_name", "underlying", "root_symbol",
    "option_root", "expiry", "strike", "option_type", "expiration_session",
    "settlement_type", "contract_multiplier",
]

def show(sec: Security) -> None:
    """Pretty-print the non-empty resolved fields of a Security."""
    print(f"symbol: {sec.symbol!r}")
    for f in FIELDS:
        v = getattr(sec, f)
        if v not in (None, {}):
            print(f"  {f:<20} {v}")
    if sec.metadata:
        print(f"  {'metadata':<20} {sec.metadata}")
    print()

def resolve(symbol, security_type=SecurityType.UNKNOWN):
    return chain.resolve(Security(symbol=symbol, security_type=security_type))

## 1. Equities & ETFs

A bare ticker is ambiguous (it could be a future root), so the equity resolver
only fires when `security_type` is `STOCK`/`ETF` — supply it.

In [3]:
show(resolve("AAPL", SecurityType.STOCK))
show(resolve("SPY", SecurityType.ETF))

symbol: 'AAPL'
  security_type        stock
  security_id          equity:AAPL
  short_name           AAPL
  underlying           AAPL
  root_symbol          AAPL

symbol: 'SPY'
  security_type        etf
  security_id          equity:SPY
  short_name           SPY
  underlying           SPY
  root_symbol          SPY



## 2. Listed options (OCC) — equity & index, with AM/PM + settlement rules

OCC symbols **self-detect** (no `security_type` needed). Index roots are enriched
from packaged rules: `SPXW` → PM-settled, `SPX` → AM-settled, both cash, with
`underlying` mapped to `SPX`.

In [4]:
show(resolve("AAPL  250117C00150000"))   # equity option
show(resolve("SPXW  260619P06100000"))   # index, PM / cash
show(resolve("SPX   260619C06100000"))   # index, AM / cash

symbol: 'AAPL  250117C00150000'
  security_type        option
  security_id          option:AAPL:2025-01-17:call:150
  short_name           AAPL 150C 2025-01-17
  underlying           AAPL
  option_root          AAPL
  expiry               2025-01-17
  strike               150
  option_type          call
  expiration_session   unknown
  settlement_type      unknown

symbol: 'SPXW  260619P06100000'
  security_type        option
  security_id          option:SPXW:2026-06-19:put:6100
  short_name           SPXW 6100P 2026-06-19
  underlying           SPX
  option_root          SPXW
  expiry               2026-06-19
  strike               6100
  option_type          put
  expiration_session   pm
  settlement_type      cash

symbol: 'SPX   260619C06100000'
  security_type        option
  security_id          option:SPX:2026-06-19:call:6100
  short_name           SPX 6100C 2026-06-19
  underlying           SPX
  option_root          SPX
  expiry               2026-06-19
  strike             

## 3. Futures — dated, active, continuous

`/ESM6` is a dated contract; `/ES` is the front-month (resolves to the
deterministic `future:ES:active` offline — inject `ResolverChain.with_tastytrade(session)`
for the live dated contract); `@ES` is the continuous back-adjusted series.

In [5]:
show(resolve("/ESM6", SecurityType.FUTURE))
show(resolve("/ES", SecurityType.FUTURE))
show(resolve("@ES", SecurityType.FUTURE))

symbol: '/ESM6'
  security_type        future
  security_id          future:ES:M6
  short_name           /ESM6
  underlying           ES
  root_symbol          ES

symbol: '/ES'
  security_type        future
  security_id          future:ES:active
  short_name           /ES
  underlying           ES
  root_symbol          ES

symbol: '@ES'
  security_type        future
  security_id          future:ES
  short_name           @ES
  underlying           ES
  root_symbol          ES



## 4. Options on futures — product-code capture + lossless round-trip

Futures options aren't OCC-format. The TT order symbol carries three parts:
the underlying future (`ESU6`), the **option series/product code** (`E1CN6`,
CME taxonomy — not computable from the other fields), and the option quartet.

The resolver **captures the product code** (which the platform's canonical
parser drops) and stashes it plus the native symbol in `metadata` — so the TT
order symbol round-trips off the `Security` itself, no registry lookup.

In [6]:
show(resolve("./ESU6 E1CN6 260701C6325"))   # E-mini S&P (x50, physical)
show(resolve("./CLU6 LO1U6 260815P72.5"))   # Crude Oil  (x1000, physical)

symbol: './ESU6 E1CN6 260701C6325'
  security_type        future_option
  security_id          future_option:ES:U6:2026-07-01:call:6325
  short_name           ESU6 6325C 2026-07-01
  underlying           ESU6
  root_symbol          ES
  option_root          E1CN6
  expiry               2026-07-01
  strike               6325
  option_type          call
  settlement_type      physical
  contract_multiplier  50
  metadata             {'native_symbol': './ESU6 E1CN6 260701C6325', 'product_code': 'E1CN6', 'underlying_future': 'ESU6'}

symbol: './CLU6 LO1U6 260815P72.5'
  security_type        future_option
  security_id          future_option:CL:U6:2026-08-15:put:72.5
  short_name           CLU6 72.5P 2026-08-15
  underlying           CLU6
  root_symbol          CL
  option_root          LO1U6
  expiry               2026-08-15
  strike               72.5
  option_type          put
  settlement_type      physical
  contract_multiplier  1000
  metadata             {'native_symbol': './CLU6 LO1

## 5. Crypto

`BTC-USD`, `ETH/USDT`, or run-together `BTCUSD` all normalize to
`crypto:{BASE}-{QUOTE}`.

In [7]:
show(resolve("BTC-USD", SecurityType.CRYPTO))
show(resolve("ETH/USDT", SecurityType.CRYPTO))
show(resolve("BTCUSD", SecurityType.CRYPTO))

symbol: 'BTC-USD'
  security_type        crypto
  security_id          crypto:BTC-USD
  short_name           BTC
  underlying           BTC
  root_symbol          BTC

symbol: 'ETH/USDT'
  security_type        crypto
  security_id          crypto:ETH-USDT
  short_name           ETH
  underlying           ETH
  root_symbol          ETH

symbol: 'BTCUSD'
  security_type        crypto
  security_id          crypto:BTC-USD
  short_name           BTC
  underlying           BTC
  root_symbol          BTC



## 6. One chain, every type — unified routing table

A single `ResolverChain` routes a mixed feed. (The `security_type` comes from the
broker/position feed; OCC and futures-option symbols also self-detect.)

In [8]:
import pandas as pd

MIXED = [
    ("AAPL", SecurityType.STOCK),
    ("SPY", SecurityType.ETF),
    ("AAPL  250117C00150000", SecurityType.UNKNOWN),
    ("SPXW  260619P06100000", SecurityType.UNKNOWN),
    ("/ESM6", SecurityType.FUTURE),
    ("./ESU6 E1CN6 260701C6325", SecurityType.UNKNOWN),
    ("BTC-USD", SecurityType.CRYPTO),
]
df = pd.DataFrame([
    {
        "input_symbol": sym,
        "security_type": (s := resolve(sym, st)).security_type.value,
        "security_id": s.security_id,
        "underlying": s.underlying,
        "settlement": s.settlement_type.value if s.settlement_type else "",
        "native_symbol": s.metadata.get("native_symbol", ""),
    }
    for sym, st in MIXED
])
df

,input_symbol,security_type,security_id,underlying,settlement,native_symbol
0,AAPL,stock,equity:AAPL,AAPL,,
1,SPY,etf,equity:SPY,SPY,,
2,AAPL 250117C00150000,option,option:AAPL:2025-01-17:call:150,AAPL,unknown,
3,SPXW 260619P06100000,option,option:SPXW:2026-06-19:put:6100,SPX,cash,
4,/ESM6,future,future:ES:M6,ES,,
5,./ESU6 E1CN6 260701C6325,future_option,future_option:ES:U6:2026-07-01:call:6325,ESU6,physical,./ESU6 E1CN6 260701C6325
6,BTC-USD,crypto,crypto:BTC-USD,BTC,,


## 7. Uncovered families pass through (`security_id=None`)

No resolver claims FX / bonds / OTC yet, so they come back un-identified — the
chain never raises; callers can fall back to the vendor symbol.

In [9]:
for sym, st in [("EURUSD", SecurityType.FX), ("T 4.25 2034", SecurityType.BOND)]:
    print(f"{sym:14} -> {resolve(sym, st).security_id}")

EURUSD         -> None
T 4.25 2034    -> None


---
**Takeaways**

- One `ResolverChain.default()` resolves stock / ETF / option / future /
  future-option / crypto into a stable `security_id`.
- Resolution is pure & offline — no market data, no broker, no network.
- Options-on-futures keep their native TT symbol in `metadata`, so converting
  back to a broker order symbol is a field read, not a registry lookup.
